## Track B: Multicultural Visual Reasoning

### 1. Environment Setup & OpenSearch Initialization

In [ ]:
import os
import ast
import io
import base64
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Dataset
from openai import OpenAI
from PIL import Image

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

BASE_URL = os.getenv("BASE_URL") or "https://api.novasearch.org/gemma4/v1"
API_KEY = os.getenv("API_KEY") or "nova-vl"
MODEL = os.getenv("MODEL") or "google/gemma-4-31b-it"

# Define target Track B Indices
cvqa_index_name = f"{OPENSEARCH_USER}_cvqa_project"
wiki_cache_index = f"{OPENSEARCH_USER}_wiki_cache"

# Initialize OpenSearch Client
client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress=True, 
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl=True,
    url_prefix='opensearch_v3',
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)

# Initialize OpenAI server client for Gemma-4-31B with verified fallbacks
openai_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Initialize embedding models matching your Phase 2 vector fields
print("Loading embedding models...")
#sbert_model = SentenceTransformer('all-mpnet-base-v2')       # 768 dim
#bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')     # 384 dim
#clip_model = SentenceTransformer('clip-ViT-B-32')             # 512 dim
bge_model= SentenceTransformer('BAAI/bge-m3') # 1024 dim, multilingual
print("Models loaded successfully.")

Loading embedding models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /Users/lucadavi/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Models loaded successfully.


### 2. Dataset Loading and Stratified Held-out Split

In [ ]:
print("Loading afaji/cvqa dataset from Hugging Face...")
cvqa_ds = load_dataset("afaji/cvqa", split="test")

def parse_subset_metadata(example):
    try:
        # Extract Language and Country safely from the Subset tuple string
        subset_tuple = ast.literal_eval(example['Subset'])
        example['language'] = subset_tuple[0]
        example['country'] = subset_tuple[1]
    except:
        example['language'] = "Unknown"
        example['country'] = "Unknown"
    return example

# Map metadata and filter for target evaluation languages
cvqa_ds = cvqa_ds.map(parse_subset_metadata)
target_languages = ["English", "Portuguese", "Arabic"]
cvqa_filtered = cvqa_ds.filter(lambda x: x['language'] in target_languages)

# Convert to pandas to sample 1,000 rows proportionally
df_cvqa = cvqa_filtered.to_pandas()
sampled_df = df_cvqa.groupby('language', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 334), random_state=42)
)

# Convert back to Dataset and enforce exactly 1000 items
working_dataset = Dataset.from_pandas(sampled_df).shuffle(seed=42)
if len(working_dataset) > 1000:
    working_dataset = working_dataset.select(range(1000))

# Create stratified Train (Retrieval Corpus) and Held-Out Test Set
split_ds = working_dataset.train_test_split(test_size=0.2, stratify_by_column='language', seed=42)
retrieval_corpus = split_ds['train']
test_set = split_ds['test']

print(f"Dataset Split complete: Retrieval Split size = {len(retrieval_corpus)}, Blind Test Split size = {len(test_set)}")

Loading afaji/cvqa dataset from Hugging Face...


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00010.parquet:   0%|          | 0.00/467M [00:00<?, ?B/s]

data/test-00001-of-00010.parquet:   0%|          | 0.00/476M [00:00<?, ?B/s]

data/test-00002-of-00010.parquet:   0%|          | 0.00/499M [00:00<?, ?B/s]

...

data/test-00005-of-00010.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

data/test-00006-of-00010.parquet:   0%|          | 0.00/518M [00:00<?, ?B/s]

data/test-00007-of-00010.parquet:   0%|          | 0.00/468M [00:00<?, ?B/s]

data/test-00008-of-00010.parquet:   0%|          | 0.00/512M [00:00<?, ?B/s]

data/test-00009-of-00010.parquet:   0%|          | 0.00/500M [00:00<?, ?B/s]

Indexing: Index the retrieval split (80%) of CVQA images, questions, and
answer options for your chosen languages in OpenSearch. Include language
and cultural region as metadata fields to support filtered retrieval.


In [ ]:
EMBEDDING_SIZE = 1024
cvqa_index_body = {
    "settings":{
        "index":{
            "knn": True,
            "number_of_shards": 1,
            "number_of_replicas": 0
        },
        "analysis": {
            "analyzer":{
                "multilingual_analyzer":{ #if we want to use BM25 retrieval
                    "type":"standard",
                    "stopwords": "_none_"
                }
            }
        }
    },
    "mappings":{
        "properties":{
            "question_id": {"type": "keyword"},
            "question": {"type": "text", "analyzer": "multilingual_analyzer"},
            "question_vector":{ #sbert embedding
                "type": "knn_vector",
                "dimension": EMBEDDING_SIZE,
                "method":{
                    "name": "hnsw",
                    "space_type":"cosinesimil",
                    "engine":"faiss"
                }
            },
            "answer": {"type": "keyword"},
            "language": {"type": "keyword"},
            "country": {"type": "keyword"},
            "category": {"type": "keyword"},
            "image_id": {"type": "keyword"},
            "image_caption": {"type": "text", "analyzer": "multilingual_analyzer"},
            "agent_answer_baseline": {"type": "keyword"},
            "agent_answer_augmented": {"type": "keyword"},
            "baseline_correct": {"type": "boolean"},
            "augmented_correct": {"type": "boolean"}
        }
    }
}

wiki_cache_index_body = {
    "settings":{
        "index":{
            "knn":True,
            "number_of_shards":1,
            "number_of_replicas":0
        },
        "analysis": {
            "analyzer":{
                "multilingual_analyzer":{
                    "type":"standard",
                    "stopwords": "_none_"
                }
            }
        }
    },
    "mappings":{
        "properties":{
            "doc_id": {"type": "keyword"}, #hash of title+chunk: sha256(title+chunk)
            "title": {"type": "text", "analyzer": "multilingual_analyzer"},
            "passage": {"type": "text", "analyzer": "multilingual_analyzer"},
            "passage_vector":{ #sbert embedding
                "type": "knn_vector",
                "dimension": EMBEDDING_SIZE,
                "method":{
                    "name": "hnsw",
                    "space_type":"cosinesimil",
                    "engine":"faiss"
                }
            },
            "language": {"type": "keyword"},
            "wikipedia_url": {"type": "keyword", "index": False},
            "wikipedia_title":{"type": "keyword"},
            "chunk_index": {"type": "integer"},
            "retrieved_for_question_ids": {"type": "keyword"},
            "retrieval_count": {"type": "integer"}
        }
    }
}

In [ ]:
if not client.indices.exists(index=cvqa_index_name):
    client.indices.create(index=cvqa_index_name, body=cvqa_index_body)
    print(f"Created OpenSearch index: {cvqa_index_name}")
else:
    print(f"OpenSearch index already exists: {cvqa_index_name}")
if not client.indices.exists(index=wiki_cache_index):
    client.indices.create(index=wiki_cache_index, body=wiki_cache_index_body)
    print(f"Created OpenSearch index: {wiki_cache_index}")
else:
    print(f"OpenSearch index already exists: {wiki_cache_index}")

In [ ]:
#compute embeddings and save them locally
from pathlib import Path
import json

BATCH_SIZE = 64
CHECKPOINT_DIR = Path("embedding_chkpt")
CHECKPOINT_DIR.mkdir(exist_ok=True)
EMBEDDING_PATH = Path("cvqa_question_embeddings.npz")
METADATA_PATH = Path("cvqa_retrieval_metadata.json")

all_ids = [row['question_id'] for row in retrieval_corpus]
all_texts = [row['question'] for row in retrieval_corpus]

if not EMBEDDING_PATH.exists():
    metadata = json.dump({
        "model": bge_model.__class__.__name__,
        "embedding_size": EMBEDDING_SIZE,
        "prefix_query": None,
        "prefix_passage": None,
        "normalized": True,
        "dataset": "afaji/cvqa",
        "num_items": len(all_texts),
    }, open(METADATA_PATH, "w"))

if not EMBEDDING_PATH.exists():
    for i in range(0, len(all_texts), BATCH_SIZE):
        checkpoint_path = CHECKPOINT_DIR / f"batch_{i}.npz"

        if checkpoint_path.exists():
            print(f"Batch {i} already processed, skipping.")
            continue
        batch_ids = all_ids[i:i+BATCH_SIZE]
        batch_texts = all_texts[i:i+BATCH_SIZE]

        batch_vectors= bge_model.encode(
            batch_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=BATCH_SIZE
        )

        np.savez(checkpoint_path, ids=batch_ids, vectors=batch_vectors)
        print(f"Saved batch {i} / {len(all_texts)} to checkpoint.")
if not EMBEDDING_PATH.exists():
    all_npz_ids=[]
    all_npz_vectors=[]

    for path in sorted(CHECKPOINT_DIR.glob("batch_*.npz")):
        data = np.load(path)
        all_npz_ids.extend(data['ids'])
        all_npz_vectors.extend(data['vectors'])

    np.savez(
        EMBEDDING_PATH,
        ids=np.array(all_npz_ids),
        vectors=np.array(all_npz_vectors)
    )

    print(f"All embeddings computed and saved to {EMBEDDING_PATH}")

In [ ]:
def generate_docs(dataset, embedding_path):
    data = np.load(embedding_path)
    id_to_vector =dict(zip(data['ids'], data['vectors']))

    for row in dataset:
        qid = row['question_id']
        yield{
            "_index": cvqa_index_name,
            "_id": qid,
            "_source":{
                "question_id": qid,
                "question": row['question'],
                "question_vector": id_to_vector[qid].tolist(),
                "answer": row['answer'],
                "language": row['language'],
                "country": row['country'],
                "category": row['category'],
                "image_id": row['image_id'],
                "image_caption": row.get('image_caption', None),
                "agent_answer_baseline": None,
                "agent_answer_augmented": None,
                "baseline_correct": None,
                "augmented_correct": None
            }
        }

helpers.bulk(
    client,
    generate_docs(retrieval_corpus, EMBEDDING_PATH),
    chunk_size=200,
    request_timeout=60
)

count = client.count(index=cvqa_index_name)['count']
print(f"Total documents indexed in {cvqa_index_name}: {count}")
if count != len(retrieval_corpus):
    print("Warning: Document count in OpenSearch does not match expected count from dataset.")
else:   
    print("Indexed all documents into OpenSearch successfully.")

3. Baseline evaluation: Evaluate Gemma 4 directly on the test set questions
without retrieval. Report accuracy per language and cultural group. Identify
which groups and question types show the largest failure rates.



4. Retrieval-augmented cultural reasoning: Extend the agent with two
complementary tools: (a) a retrieve_similar_questions(query,
language) tool that retrieves related CVQA image–question pairs from the
retrieval index, and (b) a WikipediaSearchTool (built into smolagents) for
on-demand cultural background — retrieved Wikipedia articles are cached
into the OpenSearch index as they are fetched, so the knowledge base
grows incrementally. Pass the retrieved context to the LVL
M alongside the
test set question and measure the accuracy improvement.

5. Gap analysis: Compare performance across your three cultural groups.
Characterise the failure modes — are errors due to visual recognition,
cultural knowledge, or language understanding?